# Forward Facing Prediction weights

From the earlier parts I have learnt:
* best temporal setup: `Expanding window` with `recency_decay = 0.03`
* for past soft labels: `constant method`, with `soft_label_weight = 0.5` and `confidence_cutoff = 0.7`

Now, I will be tuning for forward facing prediction to mimic the actual `test` prediction:
* I will not have access to any labels from `time_step = 36` onwards
* Thus, eg. leveraging all the labels in the training dataset (`time_step = 1-35`) and earlier predicted test rows (eg. `time_step = 36-39`) will be used to predict transactions in `time_step = 40`

Thus, I will need to tune the
  1) Forward Prediction Sample Weight (eg. predicted results from time_step 36 - 39)
  2) Forward Prediction Confidence Cutoff (eg. only train on rows where the prediction is 0.8 & above)
--------
Code flow:
train_soft_label_raw, steps 1–20
=> initial training set

labelled_train, step 21
=> predict and evaluate

step 21 predictions
=> added as pseudo-labels

labelled_train, step 22
=> predict and evaluate

step 22 predictions
=> added as pseudo-labels
--------
### Results

* the best weight is small, at 0.3 with a high confidence cut-off of 0.9
* this is despite the wide search of weight ranges from `0.1 - 2.0`
* this means that the forward pseudo-labelling may not be beneficial for the set up (thus the low weight of 0.3), as most forward predictions are not reliable enough to be included (thus the high confidence cut-off)
* as such, I will move forward with a one-shot prediction of time_steps 1-35, and then directly predict time_step 36-49 at once

In [ ]:

import numpy as np
import pandas as pd


from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier

In [2]:
### Load data

train_raw = pd.read_csv("train.csv", header=0)
train_soft_label_raw = pd.read_csv("train_soft_label.csv", header=0)
test_raw = pd.read_csv("test.csv", header=0)
edges_raw = pd.read_csv("txs_edgelist.csv", header=0)

# Use only genuinely observed labels for recursive backtesting.
# train_soft_label.csv preserves the original label column, so it is
# the single source used here to identify confirmed labelled rows.
labelled_train = train_soft_label_raw[
    train_soft_label_raw["label"].notna()
].copy()

print("Labelled training rows:", len(labelled_train))
print("Train time-step range:", labelled_train["time_step"].min(), "to", labelled_train["time_step"].max())

Labelled training rows: 31235
Train time-step range: 1 to 35


In [3]:
# Use the train/test time mapping to decide whether an edge endpoint is
# temporally available at a given cutoff.
node_times = pd.concat(
    [
        train_raw[["txId", "time_step"]],
        test_raw[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId").set_index("txId")["time_step"]

edges = edges_raw[["txId1", "txId2"]].copy()
edges["time_1"] = edges["txId1"].map(node_times)
edges["time_2"] = edges["txId2"].map(node_times)

print("Edges with unknown endpoint time:", edges[["time_1", "time_2"]].isna().any(axis=1).sum())

Edges with unknown endpoint time: 58437


In [4]:
def create_lightgbm():
    return LGBMClassifier(
        objective="binary",
        random_state=100,
        n_jobs=-1,
        verbosity=-1,
    )

def calculate_recency_weights(
    time_steps,
    reference_time_step,
    decay_rate,
):
    """
    Exponential recency weighting.

    More recent observations receive larger weights.
    """

    age = reference_time_step - time_steps

    weights = np.exp(
        -decay_rate * age
    )

    return weights

In [5]:
feature_cols = [
    c for c in train_raw.columns # only look at feat_xxx, no node_info here
    if c not in ["txId", "label", "time_step"] #do not include time_step, has no improvement n slightly reduced roc
]

print("Original feature count:", len(feature_cols))

Original feature count: 165


In [6]:
def temporal_node_features(cutoff_step):
    """Compute structural node features using edges available by cutoff_step."""
    usable_edges = edges[
        edges["time_1"].notna()
        & edges["time_2"].notna()
        & (edges["time_1"] <= cutoff_step)
        & (edges["time_2"] <= cutoff_step)
    ]

    in_degree = usable_edges.groupby("txId2").size().rename("in_degree")
    out_degree = usable_edges.groupby("txId1").size().rename("out_degree")

    features = pd.concat([in_degree, out_degree], axis=1).fillna(0)
    features["total_degree"] = (
        features["in_degree"] + features["out_degree"]
    )
    features["degree_ratio"] = (
        features["out_degree"]
        / features["in_degree"].replace(0, np.nan)
    )
    features["degree_ratio"] = (
        features["degree_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features.reset_index().rename(columns={"index": "txId"})


graph_feature_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
]

In [7]:
def add_temporal_graph_features(frame, cutoff_step):
    graph_features = temporal_node_features(cutoff_step)
    result = frame.merge(graph_features, on="txId", how="left")
    result[graph_feature_cols] = result[graph_feature_cols].fillna(0)
    return result

In [8]:
def evaluate_recursive_forward_configuration(
    labelled_data,
    historical_soft_label_data,
    feature_cols,
    initial_train_end=20,
    final_validation_step=35,
    recency_decay=0.03,
    forward_soft_weight=0.5,
    forward_confidence_cutoff=0.7,
    use_time_step=False,
    use_graph_features=True,
    weighting_method="constant",
):
    """
    Simulate forward prediction when labels are unavailable after
    the initial training period.

    True labels are used only for evaluation, never for training.
    """

    # `labelled_data` contains the genuinely observed labels used for
    # recursive validation. `historical_soft_label_data` contains the
    # retrospective p_illicit values for originally unlabelled rows.
    data = labelled_data.copy()

    available_steps = sorted(
        data["time_step"].dropna().unique()
    )

    validation_steps = [
        step
        for step in available_steps
        if initial_train_end < step <= final_validation_step
    ]

    # Build the initial training set from the retrospective soft-label
    # file. Only rows up to the initial cutoff may be used.
    historical = historical_soft_label_data[
        historical_soft_label_data["time_step"] <= initial_train_end
    ].copy()

    observed_mask = historical["label"].notna()
    pseudo_mask = (~observed_mask) & historical["p_illicit"].notna()

    # Apply the already-selected historical configuration:
    # constant weighting, source weight 0.5, cutoff 0.7.
    confident_pseudo_mask = pseudo_mask & (
        (historical["p_illicit"] <= 1.0 - 0.7)
        | (historical["p_illicit"] >= 0.7)
    )

    observed_rows = historical.loc[observed_mask].copy()
    observed_rows["training_label"] = observed_rows["label"].astype(int)
    observed_rows["label_source_weight"] = 1.0

    historical_pseudo_rows = historical.loc[confident_pseudo_mask].copy()
    historical_pseudo_rows["training_label"] = (
        historical_pseudo_rows["p_illicit"] >= 0.5
    ).astype(int)
    historical_pseudo_rows["label_source_weight"] = 0.5

    recursive_train = pd.concat(
        [observed_rows, historical_pseudo_rows],
        ignore_index=True,
    )

    fold_results = []
    pseudo_rows_added = []

    for validation_step in validation_steps:

        train_end_step = validation_step - 1

        # Rows for the next period to predict.
        # Their labels are kept separately for evaluation only.
        validation_rows = data[
            data["time_step"] == validation_step
        ].copy()

        if validation_rows.empty:
            continue

        y_true = validation_rows["label"].astype(int)

        if y_true.nunique() < 2:
            continue

        # Only rows from previous periods are allowed in training.
        train_fold = recursive_train[
            recursive_train["time_step"] <= train_end_step
        ].copy()

        if train_fold.empty:
            continue

        y_train = train_fold["training_label"].astype(int)

        if y_train.nunique() < 2:
            continue

        # Construct graph features using information available
        # up to the relevant prediction time.
        if use_graph_features:
            train_fold = add_temporal_graph_features(
                train_fold,
                cutoff_step=train_end_step,
            )

            validation_features = add_temporal_graph_features(
                validation_rows.copy(),
                cutoff_step=validation_step,
            )
        else:
            validation_features = validation_rows.copy()

        fold_feature_cols = list(feature_cols)

        if use_graph_features:
            fold_feature_cols += graph_feature_cols

        if use_time_step:
            fold_feature_cols += ["time_step"]

        X_train = train_fold[fold_feature_cols]
        X_valid = validation_features[fold_feature_cols]

        # Historical labelled rows and earlier pseudo-labelled rows
        # receive recency weighting.
        class_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )

        recency_weights = calculate_recency_weights(
            train_fold["time_step"].to_numpy(),
            reference_time_step=train_end_step,
            decay_rate=recency_decay,
        )

        source_weights = train_fold[
            "label_source_weight"
        ].to_numpy()

        sample_weights = (
            class_weights
            * recency_weights
            * source_weights
        )

        sample_weights = (
            sample_weights / sample_weights.mean()
        )

        model = create_lightgbm()

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weights,
        )

        predicted_probability = model.predict_proba(
            X_valid
        )[:, 1]

        roc_auc = roc_auc_score(
            y_true,
            predicted_probability,
        )

        fold_results.append({
            "validation_step": validation_step,
            "roc_auc": roc_auc,
            "n_validation": len(validation_rows),
            "positive_validation": int(y_true.sum()),
            "n_train": len(train_fold),
        })

        # Only sufficiently confident predictions are added
        # to the next training period.
        confident_mask = (
            (predicted_probability <= 1.0 - forward_confidence_cutoff)
            | (predicted_probability >= forward_confidence_cutoff)
        )

        pseudo_rows = validation_rows.loc[
            confident_mask
        ].copy()

        if not pseudo_rows.empty:

            pseudo_rows["training_label"] = (
                predicted_probability[confident_mask] >= 0.5
            ).astype(int)

            if weighting_method == "constant":
                confidence_multiplier = np.ones(
                    len(pseudo_rows)
                )

            elif weighting_method == "linear":
                confidence = np.maximum(
                    predicted_probability[confident_mask],
                    1.0 - predicted_probability[confident_mask],
                )

                confidence_multiplier = (
                    (confidence - 0.5) / 0.5
                )

            elif weighting_method == "squared":
                confidence = np.maximum(
                    predicted_probability[confident_mask],
                    1.0 - predicted_probability[confident_mask],
                )

                confidence_strength = (
                    (confidence - 0.5) / 0.5
                )

                confidence_multiplier = (
                    confidence_strength ** 2
                )

            else:
                raise ValueError(
                    "weighting_method must be "
                    "'constant', 'linear', or 'squared'"
                )

            pseudo_rows["label_source_weight"] = (
                forward_soft_weight
                * confidence_multiplier
            )

            pseudo_rows_added.append(
                len(pseudo_rows)
            )

            # The true labels are deliberately removed before
            # adding rows to the recursive training set.
            pseudo_rows = pseudo_rows.drop(
                columns=["label"],
                errors="ignore",
            )

            recursive_train = pd.concat(
                [
                    recursive_train,
                    pseudo_rows,
                ],
                ignore_index=True,
            )

    fold_results = pd.DataFrame(fold_results)

    if fold_results.empty:
        return {
            "fold_results": fold_results,
            "mean_roc_auc": np.nan,
            "median_roc_auc": np.nan,
            "std_roc_auc": np.nan,
            "worst_roc_auc": np.nan,
            "total_pseudo_rows": 0,
        }

    return {
        "fold_results": fold_results,
        "mean_roc_auc": fold_results["roc_auc"].mean(),
        "median_roc_auc": fold_results["roc_auc"].median(),
        "std_roc_auc": fold_results["roc_auc"].std(),
        "worst_roc_auc": fold_results["roc_auc"].min(),
        "total_pseudo_rows": sum(pseudo_rows_added),
    }

In [9]:
forward_soft_weights = [0.10, 0.30, 0.50, 0.70, 0.90, 1.00, 1.50, 2.00]
forward_confidence_cutoffs = [0.50, 0.60, 0.70, 0.80, 0.90]
forward_weighting_methods = ["constant", "linear", "squared"]

recursive_results = []

for weighting_method in forward_weighting_methods:
    for soft_weight in forward_soft_weights:
        for confidence_cutoff in forward_confidence_cutoffs:

            print(
                f"method={weighting_method}, "
                f"weight={soft_weight:.2f}, "
                f"cutoff={confidence_cutoff:.2f}"
            )

            result = evaluate_recursive_forward_configuration(
                labelled_data=labelled_train,
                historical_soft_label_data=train_soft_label_raw,
                feature_cols=feature_cols,
                initial_train_end=20,
                final_validation_step=35,
                recency_decay=0.03,
                forward_soft_weight=soft_weight,
                forward_confidence_cutoff=confidence_cutoff,
                use_time_step=False,
                use_graph_features=True,
                weighting_method=weighting_method,
            )

            recursive_results.append({
                "weighting_method": weighting_method,
                "forward_soft_weight": soft_weight,
                "forward_confidence_cutoff": confidence_cutoff,
                "mean_roc_auc": result["mean_roc_auc"],
                "median_roc_auc": result["median_roc_auc"],
                "std_roc_auc": result["std_roc_auc"],
                "worst_roc_auc": result["worst_roc_auc"],
                "total_pseudo_rows": result["total_pseudo_rows"],
            })

recursive_summary = (
    pd.DataFrame(recursive_results)
    .sort_values(
        ["mean_roc_auc", "worst_roc_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(recursive_summary)

method=constant, weight=0.10, cutoff=0.50
method=constant, weight=0.10, cutoff=0.60
method=constant, weight=0.10, cutoff=0.70
method=constant, weight=0.10, cutoff=0.80
method=constant, weight=0.10, cutoff=0.90
method=constant, weight=0.30, cutoff=0.50
method=constant, weight=0.30, cutoff=0.60
method=constant, weight=0.30, cutoff=0.70
method=constant, weight=0.30, cutoff=0.80
method=constant, weight=0.30, cutoff=0.90
method=constant, weight=0.50, cutoff=0.50
method=constant, weight=0.50, cutoff=0.60
method=constant, weight=0.50, cutoff=0.70
method=constant, weight=0.50, cutoff=0.80
method=constant, weight=0.50, cutoff=0.90
method=constant, weight=0.70, cutoff=0.50
method=constant, weight=0.70, cutoff=0.60
method=constant, weight=0.70, cutoff=0.70
method=constant, weight=0.70, cutoff=0.80
method=constant, weight=0.70, cutoff=0.90
method=constant, weight=0.90, cutoff=0.50
method=constant, weight=0.90, cutoff=0.60
method=constant, weight=0.90, cutoff=0.70
method=constant, weight=0.90, cuto

,weighting_method,forward_soft_weight,forward_confidence_cutoff,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,total_pseudo_rows
0,squared,0.3,0.9,0.995035,0.998011,0.006382,0.977738,11763
1,squared,0.5,0.5,0.994935,0.997572,0.006434,0.978403,12346
2,squared,0.1,0.6,0.994885,0.998381,0.006747,0.976673,12240
3,linear,0.7,0.8,0.994884,0.998331,0.006542,0.977605,12048
4,constant,0.1,0.7,0.994767,0.998241,0.006777,0.977289,12124
...,...,...,...,...,...,...,...,...
115,constant,1.5,0.6,0.961745,0.991913,0.066371,0.798388,12270
116,squared,2.0,0.9,0.961123,0.991476,0.069589,0.789030,11940
117,linear,2.0,0.8,0.957269,0.991012,0.071135,0.792125,12155
118,linear,1.0,0.7,0.956200,0.992268,0.076427,0.741972,12176


In [10]:
best_forward_config = recursive_summary.iloc[0]

display(best_forward_config)

weighting_method              squared
forward_soft_weight               0.3
forward_confidence_cutoff         0.9
mean_roc_auc                 0.995035
median_roc_auc               0.998011
std_roc_auc                  0.006382
worst_roc_auc                0.977738
total_pseudo_rows               11763
Name: 0, dtype: object